# Save Your Work

Before starting, save this notebook to your Google Drive:
1. Click **File** → **Save a copy in Drive**
2. The copy will open automatically
3. Work in the Google Drive copy from now on

---

# Joining Tables

Combine data from multiple tables using different join types.

In [ ]:
import sqlite3

conn = sqlite3.connect("bookstore.db")
cursor = conn.cursor()

## Join Types

- **INNER JOIN** — Only matching rows from both tables
- **LEFT JOIN** — All rows from left table + matching from right
- **RIGHT JOIN** — All rows from right table + matching from left
- **FULL OUTER JOIN** — All rows from both tables

## INNER JOIN — matching rows only

In [ ]:
# What did each customer buy?
cursor.execute("""
    SELECT c.name, b.title, o.order_date
    FROM customers c
    INNER JOIN orders o ON c.customer_id = o.customer_id
    INNER JOIN books b ON o.book_id = b.book_id
    ORDER BY c.name
""")

print("Customer purchases:")
for row in cursor.fetchall():
    print(f"  {row[0]} bought {row[1]} on {row[2]}")

## LEFT JOIN — all from left table

In [ ]:
# All customers and their orders (including those with no orders)
cursor.execute("""
    SELECT c.name, COUNT(o.order_id) as order_count
    FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
    GROUP BY c.name
""")

print("All customers with order counts:")
for row in cursor.fetchall():
    print(f"  {row[0]}: {row[1] if row[1] else 'no'} orders")

## Joining 3+ tables

In [ ]:
# Complete customer order details
cursor.execute("""
    SELECT
        c.name as customer,
        b.title as book,
        b.price,
        o.quantity,
        (b.price * o.quantity) as total
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN books b ON o.book_id = b.book_id
    ORDER BY o.order_date
""")

print("Order details:")
for row in cursor.fetchall():
    print(f"  {row[0]} bought {row[1]} (${row[2]}) x{row[3]} = ${row[4]:.2f}")

## Table aliases for readability

In [ ]:
# Using AS to shorten table names
cursor.execute("""
    SELECT
        c.name,
        COUNT(DISTINCT o.order_id) as num_orders,
        SUM(b.price * o.quantity) as total_spent
    FROM customers AS c
    LEFT JOIN orders AS o ON c.customer_id = o.customer_id
    LEFT JOIN books AS b ON o.book_id = b.book_id
    GROUP BY c.name
    ORDER BY total_spent DESC
""")

print("Customer spending summary:")
for row in cursor.fetchall():
    spent = row[2] if row[2] else 0
    print(f"  {row[0]}: {row[1]} orders, total ${spent:.2f}")

In [ ]:
conn.close()